In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
import uuid

df = spark.table("hack_nation.india_medical.india_facilities_raw")
print(f"Raw rows: {df.count()}")
df.printSchema()

Raw rows: 10033
root
 |-- name: string (nullable = true)
 |-- phone_numbers: string (nullable = true)
 |-- officialPhone: string (nullable = true)
 |-- email: string (nullable = true)
 |-- websites: string (nullable = true)
 |-- officialWebsite: string (nullable = true)
 |-- yearEstablished: string (nullable = true)
 |-- facebookLink: string (nullable = true)
 |-- twitterLink: string (nullable = true)
 |-- linkedinLink: string (nullable = true)
 |-- instagramLink: string (nullable = true)
 |-- address_line1: string (nullable = true)
 |-- address_line2: string (nullable = true)
 |-- address_line3: string (nullable = true)
 |-- address_city: string (nullable = true)
 |-- address_stateOrRegion: string (nullable = true)
 |-- address_zipOrPostcode: string (nullable = true)
 |-- address_country: string (nullable = true)
 |-- address_countryCode: string (nullable = true)
 |-- facilityTypeId: string (nullable = true)
 |-- operatorTypeId: string (nullable = true)
 |-- affiliationTypeIds: string

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

# Fix facilityTypeId typo
df = df.withColumn("facilityTypeId",
    F.when(F.col("facilityTypeId") == "farmacy", "pharmacy")
     .otherwise(F.col("facilityTypeId"))
)

# Cast lat/lng
df = df.withColumn("latitude",  F.col("latitude").cast(DoubleType()))
df = df.withColumn("longitude", F.col("longitude").cast(DoubleType()))

# Clean literal "null" strings, then use try_cast (tolerates garbage like "25.37807846")
df = df.withColumn("numberDoctors",
    F.when(F.col("numberDoctors").isin("null", "NULL", "None", ""), None)
     .otherwise(F.col("numberDoctors"))
)
df = df.withColumn("numberDoctors", F.expr("try_cast(numberDoctors as INT)"))

df = df.withColumn("capacity",
    F.when(F.col("capacity").isin("null", "NULL", "None", ""), None)
     .otherwise(F.col("capacity"))
)
df = df.withColumn("capacity", F.expr("try_cast(capacity as INT)"))

# Clean "null" strings in text columns
for col_name in ["description", "operatorTypeId", "officialWebsite",
                 "email", "yearEstablished", "equipment", "procedure",
                 "capability", "specialties", "affiliationTypeIds"]:
    df = df.withColumn(col_name,
        F.when(F.col(col_name).isin("null", "NULL", "None"), None)
         .otherwise(F.col(col_name))
    )

print("Cell 2 done")

Cell 2 done


In [0]:
df = df.withColumn("pin_code",
    F.when(
        F.col("address_zipOrPostcode").isin("null", "NULL", "None", ""), None
    ).otherwise(
        F.regexp_replace(F.col("address_zipOrPostcode"), r"\s+", "")
    )
)

# Flag invalid PINs (not exactly 6 digits)
df = df.withColumn("pin_code_valid",
    F.when(
        F.col("pin_code").isNull(), False
    ).when(
        F.col("pin_code").rlike(r"^\d{6}$"), True
    ).otherwise(False)
)

# Set non-valid PINs to null (keep the raw value in original column)
df = df.withColumn("pin_code",
    F.when(F.col("pin_code_valid"), F.col("pin_code")).otherwise(None)
)

In [0]:
STATE_MAP = {
    # Abbreviations
    "Ut": "Uttar Pradesh", "Up": "Uttar Pradesh", "U.p.": "Uttar Pradesh",
    "Gj": "Gujarat", "Mh": "Maharashtra", "Ka": "Karnataka",
    "Nct": "Delhi", "Ncr": "Delhi", "Nit": None,
    "J&k": "Jammu And Kashmir",
    # Misspellings
    "Tamilnadu": "Tamil Nadu", "Andhrapradesh": "Andhra Pradesh",
    "Madhyapradesh": "Madhya Pradesh", "Chattisgarh": "Chhattisgarh",
    "Uttaranchal": "Uttarakhand", "Pondicherry": "Puducherry",
    # Cities mistakenly put as state — Maharashtra
    "Nagpur": "Maharashtra", "Solapur": "Maharashtra", "Thane": "Maharashtra",
    "Beed": "Maharashtra", "Chandrapur": "Maharashtra", "Ambernath": "Maharashtra",
    "Mira Bhayander": "Maharashtra", "Chinchwad": "Maharashtra",
    "Pimpri-chinchwad": "Maharashtra", "Kalyan": "Maharashtra",
    "Navi Mumbai": "Maharashtra", "Navi Mumbai, Maharashtra": "Maharashtra",
    "Pune, Maharashtra": "Maharashtra", "Pune-411044": "Maharashtra",
    "Jalgaon District": "Maharashtra", "Durg": "Maharashtra",
    # Karnataka
    "Bangalore": "Karnataka", "Bengaluru": "Karnataka",
    "Belgaum": "Karnataka", "Udupi": "Karnataka", "Chikmagalur": "Karnataka",
    # Kerala
    "Kochi": "Kerala", "Ernakulam": "Kerala", "Thrissur": "Kerala",
    "Malappuram": "Kerala", "Malappuram, Kerala": "Kerala", "Kannur": "Kerala",
    "Pathanamthitta": "Kerala", "Palakkad": "Kerala", "Alappuzha": "Kerala",
    "Thiruvananthapuram": "Kerala", "Chittur": "Kerala",
    # Tamil Nadu
    "Thoothukudi": "Tamil Nadu", "Vellore": "Tamil Nadu", "Erode": "Tamil Nadu",
    "Salem": "Tamil Nadu", "Thanjavur": "Tamil Nadu", "Tiruvallur-602001": "Tamil Nadu",
    # Telangana
    "Hyderabad": "Telangana", "Secunderabad": "Telangana",
    "Karimnagar": "Telangana", "Telangana State": "Telangana", "Mandamarri": "Telangana",
    # Andhra Pradesh
    "Kurnool": "Andhra Pradesh", "Rajahmundry": "Andhra Pradesh",
    "Chittoor": "Andhra Pradesh", "Prakasam District": "Andhra Pradesh",
    # Haryana
    "Kurukshetra": "Haryana", "Gurugram": "Haryana", "Jhajjar": "Haryana",
    "Nuh": "Haryana", "Charkhi Dadri, Haryana": "Haryana",
    "Fatehabad, Haryana": "Haryana", "Sector 56": "Haryana",
    # Punjab
    "Amritsar": "Punjab", "Sangrur": "Punjab", "Gurdaspur": "Punjab",
    "Patiala": "Punjab", "Ludhiana": "Punjab", "Mohali": "Punjab",
    "Zirakpur": "Punjab", "Ropar": "Punjab", "Punjab Region": "Punjab",
    # Rajasthan
    "Jodhpur": "Rajasthan", "Jaipur": "Rajasthan", "Udaipur": "Rajasthan",
    "Sikar": "Rajasthan", "Churu": "Rajasthan", "Rajsamand, Rajasthan": "Rajasthan",
    "Pali-rajasthan": "Rajasthan", "Durgapura": "Rajasthan",
    # Gujarat
    "Rajkot": "Gujarat", "Surat": "Gujarat", "Mehsana": "Gujarat",
    "Bharuch": "Gujarat", "Gandhinagar": "Gujarat",
    "Surendranagar District": "Gujarat", "Veraval": "Gujarat",
    # Delhi
    "New Delhi": "Delhi", "Delhi Division": "Delhi", "Delhi Ncr": "Delhi",
    "North West Delhi": "Delhi", "West Delhi": "Delhi",
    "National Capital Territory Of Delhi": "Delhi", "Safdarjung Enclave": "Delhi",
    # Uttar Pradesh cities
    "Ghaziabad": "Uttar Pradesh", "Lucknow": "Uttar Pradesh",
    "Varanasi": "Uttar Pradesh", "Allahabad": "Uttar Pradesh",
    "Aligarh": "Uttar Pradesh", "Moradabad": "Uttar Pradesh",
    "Azamgarh": "Uttar Pradesh", "Ambedkar Nagar": "Uttar Pradesh",
    "Faizabad": "Uttar Pradesh", "Kalyanpur Kanpur": "Uttar Pradesh",
    "Gautam Buddha Nagar": "Uttar Pradesh",
    # West Bengal
    "Kolkata": "West Bengal", "Howrah": "West Bengal", "Hooghly": "West Bengal",
    "North 24 Parganas": "West Bengal", "Birbhum": "West Bengal",
    "Paschim Medinipur": "West Bengal", "Murshidabad": "West Bengal",
    "Alipurduar": "West Bengal", "Puruliya": "West Bengal",
    "Rajarhat": "West Bengal", "Durgapur": "West Bengal",
    "Chakdah": "West Bengal", "Kharagpur": "West Bengal", "Dinajpur": "West Bengal",
    "Khaira": "West Bengal",
    # Bihar
    "Gaya": "Bihar", "Saran": "Bihar", "Jehanabad, Bihar": "Bihar",
    "Sitamarhi": "Bihar", "Supaul": "Bihar", "Aurangabad-bihar": "Bihar",
    # Jharkhand
    "Bokaro": "Jharkhand", "Bokaro Steel City, Jharkhand": "Jharkhand",
    # Chhattisgarh
    "Raipur": "Chhattisgarh", "Bhilai": "Chhattisgarh", "Durg": "Chhattisgarh",
    # Madhya Pradesh
    "Singrauli": "Madhya Pradesh", "Jabalpur": "Madhya Pradesh",
    "Guna, Madhya Pradesh": "Madhya Pradesh",
    "Dhar District, Madhya Pradesh": "Madhya Pradesh", "Thatipur": "Madhya Pradesh",
    # Assam
    "Darrang": "Assam", "Golaghat": "Assam", "Silchar": "Assam",
    "Barpeta, Assam": "Assam", "Sibsagar": "Assam",
    # Uttarakhand
    "Mukteshwar": "Uttarakhand",
    # Jammu & Kashmir
    "Jammu & Kashmir": "Jammu And Kashmir", "Ganderbal": "Jammu And Kashmir",
    "Kupwara": "Jammu And Kashmir", "Anantnag": "Jammu And Kashmir",
    # Union territories
    "Daman And Diu": "Dadra And Nagar Haveli And Daman And Diu",
    "Ut Of Dadra & Nagar Haveli And Daman Diu": "Dadra And Nagar Haveli And Daman And Diu",
    # Tripura
    "West Tripura": "Tripura",
}

# Build a Spark CASE WHEN expression from the map
state_expr = F.col("address_stateOrRegion")
for dirty, clean in STATE_MAP.items():
    state_expr = F.when(
        F.col("address_stateOrRegion") == dirty,
        clean
    ).otherwise(state_expr)

df = df.withColumn("state_normalized", F.initcap(F.trim(state_expr)))

In [0]:
trust_expr = (
    F.lit(100)
    - F.when(
        (F.col("capability").contains("ICU") | F.col("capability").contains("Surgery") |
         F.col("capability").contains("Emergency")) &
        (F.col("equipment").isNull() | (F.col("equipment") == "[]")),
        F.lit(30)
    ).otherwise(F.lit(0))
    - F.when(
        F.col("capability").contains("Surgery") & F.col("numberDoctors").isNull(),
        F.lit(20)
    ).otherwise(F.lit(0))
    - F.when(
        F.col("description").isNull() &
        (F.col("capability").isNull() | (F.col("capability") == "[]")) &
        (F.col("procedure").isNull()  | (F.col("procedure")  == "[]")),
        F.lit(15)
    ).otherwise(F.lit(0))
    - F.when(
        F.col("specialties").isNull() | (F.col("specialties") == "[]"),
        F.lit(10)
    ).otherwise(F.lit(0))
).cast(IntegerType())

df = df.withColumn("trust_score", trust_expr)

df = df.withColumn("trust_flag",
    F.when(F.col("trust_score") >= 90, "VERIFIED")
     .when(F.col("trust_score") >= 70, "REVIEW")
     .otherwise("SUSPICIOUS")
)

In [0]:
def coalesce_text(*cols):
    return F.concat_ws(" | ", *[F.coalesce(F.col(c), F.lit("")) for c in cols])

df = df.withColumn("searchable_text",
    coalesce_text(
        "name", "description", "specialties",
        "procedure", "equipment", "capability",
        "address_city", "state_normalized", "pin_code", "facilityTypeId"
    )
)

In [0]:
df = df.withColumn("unique_id", F.monotonically_increasing_id().cast("string"))

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable("hack_nation.india_medical.india_facilities")

print(f"Written rows: {spark.table('hack_nation.india_medical.india_facilities').count()}")

Written rows: 10033


In [0]:
comments = {
    "unique_id":       "Unique row identifier. Required for Vector Search index primary key.",
    "pin_code":        "India 6-digit postal code (PIN). Use for geographic grouping and desert analysis. Null if invalid.",
    "pin_code_valid":  "True if pin_code is a valid 6-digit Indian PIN, False otherwise.",
    "state_normalized":"Cleaned official Indian state name. Always use this for state filtering, not address_stateOrRegion.",
    "facilityTypeId":  "Type of facility: hospital, clinic, pharmacy, dentist, doctor.",
    "trust_score":     "Agent-generated 0-100 trust score. 90+=VERIFIED, 70-89=REVIEW, <70=SUSPICIOUS.",
    "trust_flag":      "VERIFIED, REVIEW, or SUSPICIOUS based on trust_score.",
    "specialties":     "JSON array of medical specialties e.g. oncology, cardiology, dialysis, ophthalmology.",
    "procedure":       "JSON array of medical procedures offered.",
    "equipment":       "JSON array of medical equipment available.",
    "capability":      "JSON array of facility capabilities e.g. ICU, Surgery, Emergency.",
    "searchable_text": "Concatenated free-text blob for Vector Search. Do NOT use in SQL WHERE clauses.",
    "numberDoctors":   "Number of doctors. Often null — null + claimed advanced capability = suspicious.",
    "capacity":        "Bed/patient capacity. 99% null — use absence as a trust signal.",
    "operatorTypeId":  "Operator type: private, public, NGO. 43% null.",
    "description":     "Free-text facility description. Primary source for IDP extraction.",
}

for col, comment in comments.items():
    spark.sql(f"""
        ALTER TABLE hack_nation.india_medical.india_facilities
        ALTER COLUMN {col} COMMENT '{comment}'
    """)

print("All column comments applied.")

All column comments applied.


In [0]:
final = spark.table("hack_nation.india_medical.india_facilities")
total = final.count()

print(f"Total rows: {total}")
print(f"\nfacilityTypeId distribution:")
final.groupBy("facilityTypeId").count().orderBy(F.desc("count")).show()
print(f"\nTrust flag distribution:")
final.groupBy("trust_flag").count().orderBy(F.desc("count")).show()
print(f"\nTop 10 states:")
final.groupBy("state_normalized").count().orderBy(F.desc("count")).show(10)
print(f"\nBad PINs (null after clean):")
print(final.filter(F.col("pin_code").isNull()).count())

Total rows: 10033

facilityTypeId distribution:
+--------------+-----+
|facilityTypeId|count|
+--------------+-----+
|        clinic| 6011|
|      hospital| 2789|
|       dentist|  740|
|        doctor|  276|
|      pharmacy|  184|
|          NULL|   22|
|          null|    6|
|   78.76646423|    1|
|   26.86830139|    1|
|             0|    1|
|   77.43661499|    1|
|   75.74303436|    1|
+--------------+-----+


Trust flag distribution:
+----------+-----+
|trust_flag|count|
+----------+-----+
|  VERIFIED| 9157|
|    REVIEW|  802|
|SUSPICIOUS|   74|
+----------+-----+


Top 10 states:
+----------------+-----+
|state_normalized|count|
+----------------+-----+
|     Maharashtra| 1534|
|   Uttar Pradesh| 1084|
|         Gujarat|  849|
|      Tamil Nadu|  642|
|          Kerala|  620|
|     West Bengal|  505|
|       Rajasthan|  504|
|       Karnataka|  465|
|           Delhi|  459|
|       Telangana|  436|
+----------------+-----+
only showing top 10 rows

Bad PINs (null after clean):
16

In [0]:
from pyspark.sql import functions as F

# 1) Clean facilityTypeId to allowed set only
allowed = ["clinic", "hospital", "dentist", "doctor", "pharmacy"]

df2 = spark.table("hack_nation.india_medical.india_facilities") \
    .withColumn("facilityTypeId", F.trim(F.lower(F.col("facilityTypeId")))) \
    .withColumn(
        "facilityTypeId",
        F.when(F.col("facilityTypeId").isin("null", "", "none"), None)
         .when(F.col("facilityTypeId").isin(*allowed), F.col("facilityTypeId"))
         .otherwise(None)
    ) \
    .withColumn("facility_type_unknown", F.col("facilityTypeId").isNull())

# 2) Keep explicit pin quality flag
df2 = df2.withColumn("pin_quality",
    F.when(F.col("pin_code").isNull(), "MISSING_OR_INVALID").otherwise("VALID")
)

# 3) Overwrite final table
df2.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
   .saveAsTable("hack_nation.india_medical.india_facilities")

print("Patched india_facilities")
spark.table("hack_nation.india_medical.india_facilities") \
    .groupBy("facilityTypeId").count().orderBy(F.desc("count")).show(truncate=False)

Patched india_facilities
+--------------+-----+
|facilityTypeId|count|
+--------------+-----+
|clinic        |6011 |
|hospital      |2789 |
|dentist       |740  |
|doctor        |276  |
|pharmacy      |184  |
|NULL          |33   |
+--------------+-----+



In [ ]:
# ── CELL 10: DETECT AND FLAG PARSING ARTIFACTS ────────────────────────────────
# Root cause of the 33 extra rows:
#   The India CSV has multi-line descriptions. When Spark read the raw CSV some
#   newlines inside quoted description strings were not escaped, causing the parser
#   to start a new row mid-description. Those overflow lines land in the `name`
#   column and have no valid facilityTypeId or medical-structured fields.
#
# Detection rules (conservative — won't catch real facilities with null type):
#   (A) facilityTypeId IS NULL  AND  no medical structured data at all
#       (all of specialties, capability, procedure, equipment are null/empty)
#   (B) OR name contains '#' — hashtag social-media fragments like "#EyeCareDoctors"
#   (C) OR len(name) > 250  — description text overflowed into the name column

from pyspark.sql import functions as F

df3 = spark.table("hack_nation.india_medical.india_facilities")

no_medical_data = (
    (F.col("specialties").isNull() | (F.col("specialties") == "[]")) &
    (F.col("capability").isNull()  | (F.col("capability")  == "[]")) &
    (F.col("procedure").isNull()   | (F.col("procedure")   == "[]")) &
    (F.col("equipment").isNull()   | (F.col("equipment")   == "[]"))
)

is_artifact = (
    (F.col("facilityTypeId").isNull() & no_medical_data)
    | F.col("name").contains("#")
    | (F.length(F.col("name")) > 250)
)

df3 = df3.withColumn("is_parsing_artifact", is_artifact)

art_count  = df3.filter( F.col("is_parsing_artifact")).count()
real_count = df3.filter(~F.col("is_parsing_artifact")).count()
print(f"Real facilities  : {real_count}")
print(f"Parsing artifacts: {art_count}")
print("\nSample artifact rows (first 10):")
df3.filter(F.col("is_parsing_artifact")) \
    .select("name", "facilityTypeId", "trust_flag") \
    .show(10, truncate=80)


In [ ]:
# ── CELL 11: APPLY ARTIFACT CLEANUP AND OVERWRITE TABLE ───────────────────────
# Artifacts are kept in the table (not deleted) so row counts stay auditable.
# Three changes are applied:
#   1. searchable_text → blank string  (removes them from Vector Search matches)
#   2. trust_score     → 0             (forces out of high-trust calculations)
#   3. trust_flag      → 'ARTIFACT'    (backend SQL can filter with trust_flag != 'ARTIFACT')

from pyspark.sql import functions as F

df3 = df3.withColumn("searchable_text",
    F.when(F.col("is_parsing_artifact"), F.lit(""))
     .otherwise(F.col("searchable_text"))
)
df3 = df3.withColumn("trust_score",
    F.when(F.col("is_parsing_artifact"), F.lit(0))
     .otherwise(F.col("trust_score"))
)
df3 = df3.withColumn("trust_flag",
    F.when(F.col("is_parsing_artifact"), F.lit("ARTIFACT"))
     .otherwise(F.col("trust_flag"))
)

df3.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("hack_nation.india_medical.india_facilities")

# Verify
final = spark.table("hack_nation.india_medical.india_facilities")
print(f"Total rows in table : {final.count()}")

print("\ntrust_flag distribution:")
final.groupBy("trust_flag").count().orderBy(F.desc("count")).show()

clean = final.filter(~F.col("is_parsing_artifact"))
print(f"\nClean (non-artifact) rows : {clean.count()}")
print("\nClean facilityTypeId distribution:")
clean.groupBy("facilityTypeId").count().orderBy(F.desc("count")).show()

print("\nPin quality (clean only):")
clean.groupBy("pin_quality").count().orderBy(F.desc("count")).show()


In [ ]:
# ── CELL 12: RE-SYNC VECTOR SEARCH INDEX ──────────────────────────────────────
# The table update (Cell 11) blanked searchable_text for artifact rows.
# The Vector Search index must re-sync to pick up those changes.
# Delta Change Data Feed (CDF) was enabled during the original write, so an
# incremental sync should run fast (only 33 changed rows).

try:
    from databricks.vector_search.client import VectorSearchClient
    vsc = VectorSearchClient(disable_notice=True)
    index = vsc.get_index(
        endpoint_name="india-medical-vs",
        index_name="hack_nation.india_medical.india_facilities_index",
    )
    index.sync()
    print("Vector Search sync triggered successfully.")
    print("Wait for the index status to return to 'Online' before running queries.")
except Exception as e:
    print(f"SDK sync failed ({e})")
    print(
        "Manual alternative:\n"
        "  Databricks UI → Catalog → hack_nation.india_medical.india_facilities_index"
        " → 'Sync now'"
    )

# ── GENIE SPACE INSTRUCTION UPDATE (do in Databricks UI) ──────────────────────
# After this cell, open your Genie Space and append this line to the instructions:
#
#   Always add the filter: trust_flag != 'ARTIFACT' to every query.
#   Never return rows where is_parsing_artifact = true.
#
# This ensures Genie-generated SQL also excludes the artifact rows automatically.

print("\nDone. Genie space instruction update is a manual step — see comment above.")
